# 07: Experiment Design: Mobile Navigation & Discovery

**Context:** Funnel analysis (notebook 03) showed mobile converts at 4x lower than desktop. The drop-off starts from the very first funnel step: even session to product page view is significantly lower on mobile than desktop. The gap widens further at Add to Cart, but the root problem begins at discovery, not checkout. The 86% drop-off at the first step (session to product page) is a site-wide issue affecting all devices. Mobile is worse, which is why this experiment targets mobile specifically.

**Scope:** Smartphone sessions only (`device.deviceCategory = 'mobile'`). Tablet is excluded because it is the smallest device segment and its larger screen means UX friction is less acute, closer to the desktop experience.

**Intervention:** Redesign the mobile homepage navigation and category browsing to make product discovery easier, with simpler menu structure, more prominent category entry points, and faster load times.

**Hypothesis:** Improving mobile product discovery will increase the mobile product page view rate, which will in turn lift the mobile session conversion rate.

**Reference:** [GA BigQuery Export Schema](https://support.google.com/analytics/answer/3437719?hl=en)

##### *Funnel Conversion Rates by device (from notebook 3)*

![Funnel by device](../images/funnel_by_device.png)

## 1. Setup

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.cloud import bigquery
from scipy import stats

%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../credentials/bq-key.json"
client = bigquery.Client()
print("BigQuery client ready.")

BigQuery client ready.


## 2. Baseline Metrics

*Pull the mobile product page view rate (primary metric baseline) and average daily mobile session volume (for duration calculation).*

In [ ]:
query_baseline = """
SELECT
  device,
  COUNT(*) AS sessions,
  COUNTIF(did_view_product = 1) AS sessions_with_product_view,
  ROUND(COUNTIF(did_view_product = 1) / COUNT(*) * 100, 4) AS product_view_rate_pct,
  COUNTIF(did_purchase = 1) AS purchases,
  ROUND(COUNTIF(did_purchase = 1) / COUNT(*) * 100, 4) AS conversion_rate_pct
FROM (
  SELECT
    device.deviceCategory AS device,
    fullVisitorId,
    visitId,
    MAX(IF(totals.transactions >= 1, 1, 0)) AS did_purchase,
    MAX(IF(hits.eCommerceAction.action_type = '2', 1, 0)) AS did_view_product
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
  UNNEST(hits) AS hits
  WHERE trafficSource.source != 'analytics.google.com'
  GROUP BY device, fullVisitorId, visitId
)
GROUP BY device
ORDER BY sessions DESC
"""

df_baseline = client.query(query_baseline).to_dataframe()
print(df_baseline.to_string(index=False))

 device  sessions  sessions_with_product_view  product_view_rate_pct  purchases  conversion_rate_pct
desktop    647832                       93597                14.4477      10527               1.6250
 mobile    208445                       25812                12.3831        856               0.4107
 tablet     30347                        3638                11.9880        168               0.5536


In [3]:
query_daily_mobile = """
SELECT
  PARSE_DATE('%Y%m%d', date) AS date,
  COUNT(*) AS mobile_sessions
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE device.deviceCategory = 'mobile'
  AND trafficSource.source != 'analytics.google.com'
GROUP BY date
ORDER BY date
"""

df_daily = client.query(query_daily_mobile).to_dataframe()
avg_daily_mobile = df_daily['mobile_sessions'].mean()
print(f'Avg daily mobile sessions: {avg_daily_mobile:,.0f}')
print(f'Total days: {len(df_daily)}')

Avg daily mobile sessions: 570
Total days: 366


## 3. Experiment Parameters

*Define the test setup before calculating sample size. Each parameter choice has a business rationale.*

| Parameter | Value | Rationale |
|---|---|---|
| **Intervention** | Mobile homepage navigation redesign | Targets the first funnel bottleneck: session to product page |
| **Unit of randomisation** | Session | Each session independently assigned to control/treatment |
| **Primary metric** | Mobile product page view rate | Directly measures whether the navigation change moves users into the funnel |
| **Secondary metric** | Mobile revenue per session | Confirms the additional product page visitors are contributing real value, not just inflating traffic |
| **Guardrail metrics** | Desktop conversion rate, desktop revenue per session | Ensure no regression in unaffected segments; desktop should be unchanged since the intervention is mobile-only |
| **Statistical test** | Two-proportion z-test (one-tailed) | Testing for an uplift only |
| **Significance level (α)** | 0.05 | 5% false positive rate |
| **Statistical power (1-β)** | 0.80 | 80% chance of detecting a true effect |
| **Traffic split** | 50/50 control / treatment | Equal allocation maximises statistical power |
| **MDE** | 20% relative lift | e.g. baseline 10% product view rate, target 12% (10% × 1.20) |


## 4. Sample Size Calculation

*Using a two-proportion z-test. We need enough sessions in each group to reliably detect a 20% relative lift in mobile product page view rate.*

In [4]:
def sample_size_two_proportions(p_control, mde_relative, alpha=0.05, power=0.80):
    """
    Returns required sample size per group for a two-proportion z-test (one-tailed).
    p_control   : baseline rate
    mde_relative: minimum detectable effect as a relative lift (e.g. 0.20 = 20%)
    """
    p_treatment = p_control * (1 + mde_relative)
    p_avg = (p_control + p_treatment) / 2

    z_alpha = stats.norm.ppf(1 - alpha)   # z-score for significance level (1.645 at alpha=0.05)
    z_beta  = stats.norm.ppf(power)       # z-score for power (0.842 at power=0.80)

    # Formula:
    #   n = (z_alpha * sqrt(2 * p_avg * (1-p_avg))  <-- controls false positives, uses pooled rate
    #       + z_beta * sqrt(p_c*(1-p_c) + p_t*(1-p_t))) ^ 2  <-- controls false negatives
    #       / (p_t - p_c)^2   <-- effect size: smaller MDE means larger n
    n = (z_alpha * np.sqrt(2 * p_avg * (1 - p_avg)) +
         z_beta  * np.sqrt(p_control * (1 - p_control) + p_treatment * (1 - p_treatment))) ** 2 \
        / (p_treatment - p_control) ** 2

    return int(np.ceil(n)), p_treatment

p_product_view = 0.1238
mde = 0.20

n_per_group, p_treatment = sample_size_two_proportions(p_product_view, mde)

print(f'Baseline product page view rate: {p_product_view:.2%}')
print(f'Target rate (+ {mde:.0%} lift): {p_treatment:.2%}')
print(f'Required sessions per group: {n_per_group:,}')
print(f'Total sessions required: {n_per_group * 2:,}')

Baseline product page view rate: 12.38%
Target rate (+ 20% lift): 14.86%
Required sessions per group: 2,372
Total sessions required: 4,744


**Insights:**

- **At 20% MDE, only 2,372 sessions per group are needed** - feasible given 570 daily mobile sessions. The experiment does not require a large traffic commitment.

In [5]:
# Sensitivity: how does sample size change across different MDE assumptions?
mde_values = [0.10, 0.15, 0.20, 0.25, 0.30]
rows = []
for mde in mde_values:
    n, p_t = sample_size_two_proportions(p_product_view, mde)
    duration = np.ceil((n * 2) / avg_daily_mobile)
    rows.append({
        'MDE (relative)': f'{mde:.0%}',
        'Target rate': f'{p_t:.3%}',
        'Sessions per group': f'{n:,}',
        'Total sessions': f'{n*2:,}',
        'Est. duration (days)': int(duration)
    })

df_sensitivity = pd.DataFrame(rows)
print(df_sensitivity.to_string(index=False))

MDE (relative) Target rate Sessions per group Total sessions  Est. duration (days)
           10%     13.618%              9,124         18,248                    33
           15%     14.237%              4,136          8,272                    15
           20%     14.856%              2,372          4,744                     9
           25%     15.475%              1,547          3,094                     6
           30%     16.094%              1,094          2,188                     4


**Insights:**

- **20% MDE at 9 days is the practical sweet spot** - short enough to run cleanly, large enough that a real navigation improvement should be detectable.
- **Detecting a 10% lift would require 33 days** - over 4 weeks, introducing seasonal noise and making it harder to isolate the navigation change as the cause.
- **Sample size shrinks fast as MDE grows** - going from 10% to 20% MDE cuts required sessions by 74% (18,248 to 4,744). If the intervention is effective, a 20% relative lift on product page views is a reasonable minimum bar to justify a full rollout.

## 5. Experiment Duration

*Converts the required sessions into calendar weeks. Rounding up to full weeks ensures the experiment captures at least one complete day-of-week cycle, avoiding bias from weekday/weekend traffic patterns.*

In [6]:
mde_chosen = 0.20
n_per_group, _ = sample_size_two_proportions(p_product_view, mde_chosen)
total_sessions  = n_per_group * 2
duration_raw    = total_sessions / avg_daily_mobile
duration_weeks  = int(np.ceil(duration_raw / 7))   # round up to full weeks

print(f'Avg daily mobile sessions : {avg_daily_mobile:,.0f}')
print(f'Total sessions needed     : {total_sessions:,}')
print(f'Raw duration              : {duration_raw:.1f} days')
print(f'Recommended duration      : {duration_weeks} week(s) ({duration_weeks * 7} days)')

Avg daily mobile sessions : 570
Total sessions needed     : 4,744
Raw duration              : 8.3 days
Recommended duration      : 2 week(s) (14 days)


**Insights:**

- **8.3 raw days rounds up to 2 weeks (14 days)**. The extra days help ensure both control and treatment are exposed to the same weekday/weekend mix, which matters here because funnel analysis showed conversion intent is nearly double on weekdays vs weekends.
- **2 weeks is a short and practical window** - low risk of external events contaminating the result during the experiment period.

## 6. Pre-launch Checklist

*Not executable here. Apply these checks once the experiment is live.*

**Sample Ratio Mismatch (SRM)**

In a 50/50 split, control and treatment should receive roughly equal session counts. SRM occurs when the observed split is significantly unequal, for example 6,000 sessions in control and 4,000 in treatment, indicating the randomisation mechanism is broken. Common causes include cookies resetting mid-experiment, bots being included, or a bug in the assignment logic.

To check: run a chi-squared test comparing observed counts to the expected 50/50 split. The chi-squared statistic measures how far the observed distribution deviates from expected. If p < 0.05, the split is too unequal to trust and results should be discarded regardless of what they show.

```python
from scipy.stats import chisquare
observed = [n_control, n_treatment]
expected = [total / 2, total / 2]
chi2, p = chisquare(observed, f_exp=expected)
# p < 0.05 means SRM detected
```

Run this check after the first 24-48 hours, not at the end. Catching SRM early avoids wasting 2 weeks on a broken experiment.

**Novelty effect**

Mobile users may behave differently simply because the navigation looks new, not because it is better. Segment results by new vs returning visitors: if returning visitors drive all the lift and new visitors show nothing, the effect is likely novelty-driven and may decay after rollout.

## 7. Results Analysis

*Not executable here. Run once the experiment has collected sufficient data.*

**Statistical test: two-proportion z-test (one-tailed)**

After the experiment runs for 2 weeks, compare the product page view rate between control and treatment using a two-proportion z-test. The z-statistic measures how many standard deviations the observed difference is from zero. The p-value answers whether that difference is explainable by chance.

Interpret results as follows:
- p < 0.05 and lift is positive: statistically significant uplift, the navigation change worked
- p >= 0.05: insufficient evidence of a lift, do not ship
- Check guardrail metrics (desktop conversion rate, desktop revenue per session) regardless of the primary result: if either regressed, investigate before any rollout decision

```python
def analyse_experiment(n_control, conv_control, n_treatment, conv_treatment, alpha=0.05):
    p_c    = conv_control / n_control
    p_t    = conv_treatment / n_treatment
    p_pool = (conv_control + conv_treatment) / (n_control + n_treatment)

    se      = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))
    z       = (p_t - p_c) / se
    p_value = 1 - stats.norm.cdf(z)
    lift    = (p_t - p_c) / p_c

    print(f'Control   : {n_control:,} sessions | {conv_control:,} product views | {p_c:.3%}')
    print(f'Treatment : {n_treatment:,} sessions | {conv_treatment:,} product views | {p_t:.3%}')
    print(f'Relative lift : {lift:+.1%}')
    print(f'P-value       : {p_value:.4f}')

# replace with actual experiment results
analyse_experiment(n_control=..., conv_control=..., n_treatment=..., conv_treatment=...)
```

## 8. Summary

**Experiment:** Mobile homepage navigation redesign, targeting the first funnel bottleneck identified in funnel analysis.

**Design:**
- Intervention: simpler mobile navigation, more prominent category entry points
- Unit: session-level randomisation, mobile (smartphone) traffic only
- Primary metric: mobile product page view rate
- Secondary metric: mobile revenue per session
- Guardrail metrics: desktop conversion rate, desktop revenue per session
- MDE: 20% relative lift on product page view rate
- α = 0.05, power = 0.80, 50/50 split

**Key outputs:**
- Baseline mobile product page view rate: 12.38%
- Target rate at 20% lift: 14.86%
- Required sessions per group: 2,372 (total: 4,744)
- Avg daily mobile sessions: 570
- Recommended duration: 2 weeks

**Decision rules:**
- Ship if: p-value < 0.05 on primary metric and guardrail metrics show no regression
- Hold if: SRM detected, or experiment ran for fewer than one full week
- Primary up, revenue per session flat: more users browsed but did not buy; the downstream funnel may need a separate intervention
- Primary flat: navigation redesign did not change user behaviour; revisit the intervention